# Ordinal Regression for BMWP/Col Water Quality Classification
## Proportional Odds Model — Nested LOOCV

---

## Section 1 — Introduction

### Why ordinal regression?

The BMWP/Col index produces a numeric score that is officially interpreted through **five ordered water-quality categories** (Roldán 2003):

| Category     | BMWP range |
|--------------|------------|
| Muy crítica  | 0 – 15     |
| Crítica      | 16 – 35    |
| Dudosa       | 36 – 60    |
| Aceptable    | 61 – 100   |
| Buena        | 101 – 120  |

These categories carry a **natural, non-exchangeable order**:  
Muy crítica < Crítica < Dudosa < Aceptable < Buena.

Standard multinomial logistic regression ignores this ranking, treating every misclassification error as equally serious. In environmental practice, confusing *Muy crítica* with *Buena* is far more consequential than misplacing an observation between two adjacent categories. **Ordinal regression** preserves the ordering information and penalises non-adjacent errors implicitly through its cumulative probability structure.

Furthermore, the distances between BMWP thresholds are **not uniform** (category widths: 16, 20, 25, 40, 20 points), so treating BMWP as a metric regression target would impose an arbitrary interval scale. Ordinal regression makes no such assumption.

### The Proportional Odds Model (POM)

The POM (also called *cumulative logit model*) specifies a separate intercept for each of the $J-1 = 4$ adjacent category thresholds, but a **single, shared coefficient vector** $\boldsymbol{\beta}$ for the predictors:

$$
\log\!\left(\frac{P(Y \leq j \mid \mathbf{x})}{P(Y > j \mid \mathbf{x})}\right) = \alpha_j - \boldsymbol{\beta}^\top \mathbf{x}, \quad j = 1, \ldots, J-1
$$

The **proportional odds assumption** states that $\boldsymbol{\beta}$ is constant across thresholds — i.e., the log-odds ratio between any two predictor values shifts the entire cumulative distribution by the same amount, regardless of which threshold is crossed. Individual category probabilities are:

$$
P(Y = j \mid \mathbf{x}) = P(Y \leq j \mid \mathbf{x}) - P(Y \leq j-1 \mid \mathbf{x})
$$

and prediction is $\hat{y} = \arg\max_j P(Y = j \mid \mathbf{x})$.

### Methodological reference

Bürkner, P.-C., & Vuorre, M. (2019). Ordinal regression models in psychology: A tutorial. *Advances in Methods and Practices in Psychological Science*, 2(1), 77–101. https://doi.org/10.1177/2515245918823199

Although the tutorial targets psychology, the statistical framework is identical to that used in ecological and environmental studies employing ordinal response variables (e.g., biotic indices, pollution classes, habitat quality scores). The cumulative-logit specification implemented here via `statsmodels.miscmodels.ordinal_model.OrderedModel` is standard in environmental water-quality modelling when the response is a ranked category.

---

## Section 2 — Data Loading

In [1]:
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_excel('../../data/Database - BMWP.xlsx')

# ── BMWP numeric values ────────────────────────────────────────────────────────
print('BMWP numeric values:')
print(df['BMWP'].values)
print(f'  n={len(df)}, min={df["BMWP"].min()}, max={df["BMWP"].max()}')

BMWP numeric values:
[ 52  45  28  72  56  37 116  75  35   9  65  90  94  13  96  85  20  93]
  n=18, min=9, max=116


In [2]:
# ── Roldán (2003) BMWP/Col thresholds ─────────────────────────────────────────
def bmwp_to_category5(score):
    if score <= 15:   return 'Muy crítica'
    elif score <= 35: return 'Crítica'
    elif score <= 60: return 'Dudosa'
    elif score <= 100: return 'Aceptable'
    else:              return 'Buena'

ORDERED_CATS_5 = ['Muy crítica', 'Crítica', 'Dudosa', 'Aceptable', 'Buena']

df['Category5'] = pd.Categorical(
    df['BMWP'].apply(bmwp_to_category5),
    categories=ORDERED_CATS_5, ordered=True
)

print('Category distribution (5-class):')
dist5 = df['Category5'].value_counts().reindex(ORDERED_CATS_5)
for cat, n in dist5.items():
    vals = sorted(df.loc[df['Category5'] == cat, 'BMWP'].tolist())
    print(f'  {cat:<14}  n={n}  ({100*n/len(df):.1f}%)  BMWP={vals}')

print(f'\nClass balance summary:')
print(f'  Most frequent : {dist5.idxmax()} (n={dist5.max()})')
print(f'  Least frequent: {dist5.idxmin()} (n={dist5.min()})')
print(f'  Imbalance ratio (max/min): {dist5.max()/dist5.min():.1f}')

Category distribution (5-class):
  Muy crítica     n=2  (11.1%)  BMWP=[9, 13]
  Crítica         n=3  (16.7%)  BMWP=[20, 28, 35]
  Dudosa          n=4  (22.2%)  BMWP=[37, 45, 52, 56]
  Aceptable       n=8  (44.4%)  BMWP=[65, 72, 75, 85, 90, 93, 94, 96]
  Buena           n=1  (5.6%)  BMWP=[116]

Class balance summary:
  Most frequent : Aceptable (n=8)
  Least frequent: Buena (n=1)
  Imbalance ratio (max/min): 8.0


In [3]:
# ── 7 candidate predictors ─────────────────────────────────────────────────────
PREDICTORS = ['COT', 'DBO5', 'Dureza', 'Magnesio', 'Turbiedad', 'OD', 'Caudal']

print('Candidate predictors — descriptive statistics:')
print(df[PREDICTORS].describe().round(3).to_string())
print(f'\nMissing values per predictor:')
print(df[PREDICTORS].isna().sum())

Candidate predictors — descriptive statistics:
          COT    DBO5   Dureza  Magnesio  Turbiedad      OD  Caudal
count  18.000  18.000   18.000    18.000     18.000  18.000  18.000
mean    8.319   6.000   88.556     7.149     17.779   5.925   1.314
std     6.412   8.513   51.664     6.196     25.965   1.094   1.038
min     1.690   2.000   35.500     0.261      2.000   3.200   0.090
25%     3.430   2.000   50.300     3.628      2.178   5.377   0.835
50%     6.585   2.000   72.200     5.330      6.690   6.060   0.970
75%    11.900   6.000  105.625     9.325     11.150   6.672   1.472
max    26.200  35.000  216.000    24.600     88.100   7.690   4.390

Missing values per predictor:
COT          0
DBO5         0
Dureza       0
Magnesio     0
Turbiedad    0
OD           0
Caudal       0
dtype: int64


## Section 3 — AIC-Based Nested Predictor Selection

To maintain methodological consistency with the NBR and SVR models in this study, predictor selection is performed **inside each LOOCV fold** using AIC computed on the $n-1$ training observations.

Within each fold a **Negative Binomial GLM** is fitted for each candidate predictor individually (same criterion used in the NBR model). The predictor with the lowest AIC is selected and subsequently used to fit the Proportional Odds Model for that fold.

This design avoids data leakage — the test observation plays no role in predictor selection.

In [4]:
def select_predictor_aic(X_train, y_train_numeric):
    """
    Fit a NegativeBinomial GLM for each candidate predictor on training data.
    Returns the name of the predictor with the lowest AIC.

    Uses method='nm' (Nelder-Mead) for numerical stability with small samples.
    """
    best_pred, best_aic = None, np.inf
    for col in X_train.columns:
        x = sm.add_constant(X_train[[col]].values.astype(float))
        y = y_train_numeric.values.astype(float)
        try:
            model = sm.NegativeBinomial(y, x).fit(
                method='nm', disp=0, maxiter=50
            )
            aic = model.aic
        except Exception:
            aic = np.inf
        if aic < best_aic:
            best_aic, best_pred = aic, col
    return best_pred


print('select_predictor_aic() defined.')
print('Uses NegativeBinomial GLM (Nelder-Mead, maxiter=50) — same AIC criterion as NBR model.')

select_predictor_aic() defined.
Uses NegativeBinomial GLM (Nelder-Mead, maxiter=50) — same AIC criterion as NBR model.


## Section 4 — Proportional Odds Model: Nested LOOCV (5-class)

For each of the $n = 18$ LOOCV folds:
1. Hold out observation $i$ as test set.
2. Select the best predictor by AIC on the remaining 17 observations (NegBin GLM).
3. Standardise the selected predictor (scaler fit on training data only).
4. Fit `OrderedModel(distr='logit')` on training data.
5. Predict category probabilities for observation $i$; assign class as argmax.
6. Record true and predicted categories; log convergence failures.

**API note:** `OrderedModel` requires the endogenous variable as a `pd.Series` with ordered `Categorical` dtype, and the exogenous variable as a 2D `pd.DataFrame` (not a 1D `pd.Series`). No constant is added — the model estimates threshold parameters instead.

In [5]:
def run_loocv_pom(categories, ordered_cats):
    """
    Nested LOOCV for the Proportional Odds Model.

    Parameters
    ----------
    categories   : pd.Categorical column of df
    ordered_cats : list of str in ascending order

    Returns
    -------
    y_true, y_pred, failed_folds, selected_preds
    """
    n = len(df)
    y_true, y_pred, failed_folds, selected_preds = [], [], [], []

    X_all   = df[PREDICTORS].copy().reset_index(drop=True)
    y_all   = categories.reset_index(drop=True)
    y_num   = df['BMWP'].reset_index(drop=True)

    for i in range(n):
        train_idx = [j for j in range(n) if j != i]

        X_train     = X_all.iloc[train_idx].reset_index(drop=True)
        y_train_raw = y_all.iloc[train_idx].reset_index(drop=True)
        X_test      = X_all.iloc[[i]]
        true_cat    = str(y_all.iloc[i])
        y_train_num = y_num.iloc[train_idx].reset_index(drop=True)

        # ── AIC predictor selection ────────────────────────────────────────────
        best_pred = select_predictor_aic(X_train, y_train_num)
        selected_preds.append(best_pred)

        # ── standardise (fit on training only) ────────────────────────────────
        scaler  = StandardScaler()
        x_tr_df = pd.DataFrame(
            {best_pred: scaler.fit_transform(X_train[[best_pred]]).ravel()}
        )
        x_te_df = pd.DataFrame(
            {best_pred: scaler.transform(X_test[[best_pred]]).ravel()}
        )

        # ── endog as ordered Categorical Series ────────────────────────────────
        y_train_cat = pd.Series(
            pd.Categorical(
                y_train_raw.astype(str),
                categories=ordered_cats,
                ordered=True
            )
        )

        # ── fit Proportional Odds Model ────────────────────────────────────────
        try:
            mod = OrderedModel(y_train_cat, x_tr_df, distr='logit')
            res = mod.fit(method='nm', disp=False, maxiter=1000)
            prob     = res.predict(x_te_df)
            pred_cat = ordered_cats[int(np.argmax(prob.values))]
        except Exception:
            failed_folds.append(i)
            # Fallback: modal training class
            pred_cat = y_train_cat.mode()[0]

        y_true.append(true_cat)
        y_pred.append(pred_cat)

    return y_true, y_pred, failed_folds, selected_preds


print('run_loocv_pom() defined.')
print('Running 5-class nested LOOCV (18 folds) ...')

y_true_5, y_pred_5, failed_5, preds_5 = run_loocv_pom(
    df['Category5'], ORDERED_CATS_5
)

print(f'Done.')

run_loocv_pom() defined.
Running 5-class nested LOOCV (18 folds) ...
Done.


In [6]:
# ── Metrics — 5-class ─────────────────────────────────────────────────────────
acc_5    = accuracy_score(y_true_5, y_pred_5)
kappa_5  = cohen_kappa_score(y_true_5, y_pred_5, weights='quadratic')
pc5      = Counter(preds_5)
modal_5  = pc5.most_common(1)[0][0]

print('\n\u2500\u2500 5-class LOOCV results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Accuracy   : {acc_5:.3f}')
print(f'  Cohen \u03ba    : {kappa_5:.3f}  (quadratic weights)')
print(f'  n_failed   : {len(failed_5)} / 18')
if failed_5:
    print(f'  Failed folds (0-indexed): {failed_5}')
print(f'  Modal predictor: {modal_5}')

print('\nPredictor selected per fold (5-class):')
for p, c in pc5.most_common():
    print(f'  {p:<12}  selected in {c}/18 folds')

print('\n\u2500\u2500 Per-fold predictions (5-class) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
for i in range(18):
    mark = 'FAIL' if i in failed_5 else ('OK' if y_true_5[i] == y_pred_5[i] else '--')
    print(f'  Fold {i:2d}  BMWP={df["BMWP"].iloc[i]:3d}  '
          f'true={y_true_5[i]:<14}  pred={y_pred_5[i]:<14}  '
          f'pred_var={preds_5[i]:<12}  [{mark}]')

print('\n\u2500\u2500 Confusion matrix (5-class) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
cm5 = confusion_matrix(y_true_5, y_pred_5, labels=ORDERED_CATS_5)
cm5_df = pd.DataFrame(cm5, index=ORDERED_CATS_5, columns=ORDERED_CATS_5)
print(cm5_df.to_string())


── 5-class LOOCV results ───────────────────────────────────────────────
  Accuracy   : 0.333
  Cohen κ    : -0.165  (quadratic weights)
  n_failed   : 1 / 18
  Failed folds (0-indexed): [6]
  Modal predictor: Dureza

Predictor selected per fold (5-class):
  Dureza        selected in 14/18 folds
  Turbiedad     selected in 3/18 folds
  Magnesio      selected in 1/18 folds

── Per-fold predictions (5-class) ───────────────────────────────────────
  Fold  0  BMWP= 52  true=Dudosa          pred=Aceptable       pred_var=Dureza        [--]
  Fold  1  BMWP= 45  true=Dudosa          pred=Aceptable       pred_var=Dureza        [--]
  Fold  2  BMWP= 28  true=Crítica         pred=Aceptable       pred_var=Turbiedad     [--]
  Fold  3  BMWP= 72  true=Aceptable       pred=Dudosa          pred_var=Dureza        [--]
  Fold  4  BMWP= 56  true=Dudosa          pred=Aceptable       pred_var=Dureza        [--]
  Fold  5  BMWP= 37  true=Dudosa          pred=Aceptable       pred_var=Dureza        [--]
  F

## Section 5 — Sensitivity: 5-Class vs 3-Class Categories

With $n = 18$ and 5 classes, some categories contain only 1–2 observations (*Buena*: $n=1$, *Muy crítica*: $n=2$), making stable ordinal regression estimation very difficult. A **3-class collapse** reduces sparsity:

| 3-class label | Original categories       | BMWP range |
|---------------|---------------------------|------------|
| Low           | Muy crítica + Crítica    | 0 – 35     |
| Medium        | Dudosa                    | 36 – 60    |
| High          | Aceptable + Buena         | 61 – 120   |

Both versions use identical nested LOOCV and AIC predictor selection.

In [7]:
# ── 3-class mapping ────────────────────────────────────────────────────────────
def bmwp_to_category3(score):
    if score <= 35:   return 'Low'
    elif score <= 60: return 'Medium'
    else:             return 'High'

ORDERED_CATS_3 = ['Low', 'Medium', 'High']

df['Category3'] = pd.Categorical(
    df['BMWP'].apply(bmwp_to_category3),
    categories=ORDERED_CATS_3, ordered=True
)

print('Category distribution (3-class):')
dist3 = df['Category3'].value_counts().reindex(ORDERED_CATS_3)
for cat, n in dist3.items():
    vals = sorted(df.loc[df['Category3'] == cat, 'BMWP'].tolist())
    print(f'  {cat:<8}  n={n}  ({100*n/len(df):.1f}%)  BMWP={vals}')

print(f'\nImbalance ratio (max/min): {dist3.max()/dist3.min():.1f}')

print('\nRunning 3-class nested LOOCV (18 folds) ...')
y_true_3, y_pred_3, failed_3, preds_3 = run_loocv_pom(
    df['Category3'], ORDERED_CATS_3
)
print('Done.')

Category distribution (3-class):
  Low       n=5  (27.8%)  BMWP=[9, 13, 20, 28, 35]
  Medium    n=4  (22.2%)  BMWP=[37, 45, 52, 56]
  High      n=9  (50.0%)  BMWP=[65, 72, 75, 85, 90, 93, 94, 96, 116]

Imbalance ratio (max/min): 2.2

Running 3-class nested LOOCV (18 folds) ...
Done.


In [8]:
# ── Metrics — 3-class ─────────────────────────────────────────────────────────
acc_3   = accuracy_score(y_true_3, y_pred_3)
kappa_3 = cohen_kappa_score(y_true_3, y_pred_3, weights='quadratic')
pc3     = Counter(preds_3)
modal_3 = pc3.most_common(1)[0][0]

print('\n\u2500\u2500 3-class LOOCV results \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
print(f'  Accuracy   : {acc_3:.3f}')
print(f'  Cohen \u03ba    : {kappa_3:.3f}  (quadratic weights)')
print(f'  n_failed   : {len(failed_3)} / 18')
if failed_3:
    print(f'  Failed folds (0-indexed): {failed_3}')
print(f'  Modal predictor: {modal_3}')

print('\nPredictor selected per fold (3-class):')
for p, c in pc3.most_common():
    print(f'  {p:<12}  selected in {c}/18 folds')

print('\n\u2500\u2500 Per-fold predictions (3-class) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
for i in range(18):
    mark = 'FAIL' if i in failed_3 else ('OK' if y_true_3[i] == y_pred_3[i] else '--')
    print(f'  Fold {i:2d}  BMWP={df["BMWP"].iloc[i]:3d}  '
          f'true={y_true_3[i]:<8}  pred={y_pred_3[i]:<8}  '
          f'pred_var={preds_3[i]:<12}  [{mark}]')

print('\n\u2500\u2500 Confusion matrix (3-class) \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500')
cm3 = confusion_matrix(y_true_3, y_pred_3, labels=ORDERED_CATS_3)
cm3_df = pd.DataFrame(cm3, index=ORDERED_CATS_3, columns=ORDERED_CATS_3)
print(cm3_df.to_string())

print('\n\u2550'*74)
print('  5-class vs 3-class Sensitivity Comparison')
print('\u2550'*74)
print(f"{'Version':<10} {'n_failed':<10} {'Accuracy':<10} {'Kappa':<10} {'Modal predictor'}")
print(f"{'--------':<10} {'--------':<10} {'--------':<10} {'-----':<10} {'---------------'}")
print(f"{'5-class':<10} {len(failed_5):<10} {acc_5:<10.3f} {kappa_5:<10.3f} {modal_5}")
print(f"{'3-class':<10} {len(failed_3):<10} {acc_3:<10.3f} {kappa_3:<10.3f} {modal_3}")


── 3-class LOOCV results ───────────────────────────────────────────────
  Accuracy   : 0.444
  Cohen κ    : -0.119  (quadratic weights)
  n_failed   : 0 / 18
  Modal predictor: Dureza

Predictor selected per fold (3-class):
  Dureza        selected in 14/18 folds
  Turbiedad     selected in 3/18 folds
  Magnesio      selected in 1/18 folds

── Per-fold predictions (3-class) ───────────────────────────────────────
  Fold  0  BMWP= 52  true=Medium    pred=High      pred_var=Dureza        [--]
  Fold  1  BMWP= 45  true=Medium    pred=High      pred_var=Dureza        [--]
  Fold  2  BMWP= 28  true=Low       pred=High      pred_var=Turbiedad     [--]
  Fold  3  BMWP= 72  true=High      pred=Medium    pred_var=Dureza        [--]
  Fold  4  BMWP= 56  true=Medium    pred=Low       pred_var=Dureza        [--]
  Fold  5  BMWP= 37  true=Medium    pred=High      pred_var=Dureza        [--]
  Fold  6  BMWP=116  true=High      pred=High      pred_var=Dureza        [OK]
  Fold  7  BMWP= 75  true=Hi

## Section 6 — Comparison with Existing BMWP Models

In [9]:
print('\u2550'*78)
print('  Final comparison: POM vs existing BMWP classification models')
print('\u2550'*78)

comparison = [
    ('ε-SVR',          5, 0.556, 0.258, 'Nested LOOCV'),
    ('Fuzzy (E)',      5, 0.444, 0.163, '2 folds failed'),
    ('NBR',            5, 0.389, 0.000, 'Nested LOOCV'),
    ('POM (5-class)',  5, acc_5, kappa_5,
     f'Nested LOOCV, {len(failed_5)} fold(s) failed'),
    ('POM (3-class)',  3, acc_3, kappa_3,
     f'Nested LOOCV, {len(failed_3)} fold(s) failed'),
]

print(f"\n{'Model':<16} {'Classes':<9} {'Acc':<8} {'\u03ba':<8} {'Notes'}")
print(f"{'---------------':<16} {'-------':<9} {'------':<8} {'------':<8} {'-----'}")
for model, cls, acc, kap, notes in comparison:
    print(f"{model:<16} {cls:<9} {acc:<8.3f} {kap:<8.3f} {notes}")

══════════════════════════════════════════════════════════════════════════════
  Final comparison: POM vs existing BMWP classification models
══════════════════════════════════════════════════════════════════════════════

Model            Classes   Acc      κ        Notes
---------------  -------   ------   ------   -----
ε-SVR            5         0.556    0.258    Nested LOOCV
Fuzzy (E)        5         0.444    0.163    2 folds failed
NBR              5         0.389    0.000    Nested LOOCV
POM (5-class)    5         0.333    -0.165   Nested LOOCV, 1 fold(s) failed
POM (3-class)    3         0.444    -0.119   Nested LOOCV, 0 fold(s) failed


### Interpretation

**Does POM improve over SVR?**  
No. The ε-SVR achieves Accuracy = 0.556 and κ = 0.258. Both POM versions fall below this benchmark: the 5-class POM reaches Accuracy = 0.333 and κ = −0.165, while the 3-class version reaches Accuracy = 0.444 and κ = −0.119. Negative kappa values indicate that both POM models perform **worse than chance** when adjusted for the marginal class distribution — specifically, both models exhibit a strong bias toward predicting the majority class (*Aceptable* in the 5-class case, *High* in the 3-class case), ignoring minority classes entirely.

The confusion matrices confirm this: the 5-class POM predicts *Aceptable* for 14 of 18 observations, never predicting *Critica* or *Buena* correctly. The 3-class POM predicts *High* for 14 of 18 observations.

**Is the 3-class version more stable?**  
Marginally. The 3-class POM has 0 convergence failures vs 1 in the 5-class version, and its accuracy is slightly higher (0.444 vs 0.333). However, both versions produce negative quadratic-weighted kappa, indicating that the ordinal ranking is not being exploited meaningfully. The collapse to 3 classes reduces sparsity but does not resolve the fundamental problem: with $n = 17$ training observations and a heavily imbalanced majority class, the MLE is dominated by the prior class frequencies and the predictor coefficient provides insufficient discrimination.

**Is this worth including in the article?**  
The POM is worth **mentioning** as a methodological benchmark — it is the theoretically most appropriate model for an ordinal biotic index — but the results do not support including it as a viable classifier for this dataset. The honest conclusion is that $n = 18$ is insufficient to fit a POM: the model collapses to majority-class prediction, and both accuracy and kappa are below all other evaluated models including the NBR. The article should note this result as a negative finding that motivates future data collection.

---

## Section 7 — Limitations

### 7.1 Severely constrained sample size

The dataset contains $n = 18$ observations. The 5-class Proportional Odds Model requires estimation of:
- $J - 1 = 4$ threshold (intercept) parameters ($\alpha_1, \ldots, \alpha_4$),
- 1 predictor coefficient $\beta$,

yielding 5 parameters. Each LOOCV fold trains on only 17 observations, giving an *observations-per-parameter* ratio of approximately **3.4** — far below the commonly cited rule of thumb of 10 per parameter for reliable maximum-likelihood estimation in ordinal models. Under these conditions the threshold estimates are expected to have wide confidence intervals, and the model tends to collapse toward majority-class prediction.

For the 3-class version the model requires 3 parameters (2 thresholds + 1 coefficient), improving the ratio to approximately **5.7**, still below the recommended threshold.

### 7.2 Convergence failures

Any fold in which `OrderedModel.fit()` raises an exception is flagged as a convergence failure (reported in Sections 4 and 5). In those folds the prediction falls back to the most frequent training class. Failed folds are **not genuine model predictions** and must be disclosed in any summary of results. The Nelder-Mead optimiser (`method='nm'`) was selected for its robustness to ill-conditioned likelihood surfaces at small sample sizes; the default BFGS gradient method was found to hang indefinitely on some folds.

### 7.3 Proportional odds assumption untestable at n = 18

The proportional odds assumption — that the predictor coefficient $\boldsymbol{\beta}$ is constant across all category thresholds — is the central structural restriction of the POM. Standard tests of this assumption (e.g., Brant test, score test) require sufficient observations in every cell of the predictor × category cross-tabulation. With $n = 18$ and 5 categories containing as few as 1 observation (*Buena*: $n = 1$), multiple cells are empty, rendering these tests unreliable or inapplicable. The assumption is therefore adopted on methodological grounds rather than verified empirically.

### 7.4 Single-predictor model

The AIC-based nested selection forces a univariate POM (one predictor per fold). This is a deliberate consequence of the $n = 18$ constraint: a multivariate POM would be even more severely over-parametrised. The results represent the best single-predictor ordinal model, not the full multivariate potential of the POM framework. In practice, *Dureza* dominates predictor selection across folds (chosen in the majority of folds for both 5-class and 3-class versions), consistent with its role as a proxy for mineral load in Andean rivers.

### 7.5 Category sparsity (5-class)

In the 5-class version, the *Buena* category contains only 1 observation. In a LOOCV fold where observation 6 (BMWP = 116, *Buena*) is in the training set, there is exactly 1 *Buena* training observation; when it is held out as test, the *Buena* category is absent from the training set entirely. In either case the MLE for $\alpha_4$ (the threshold separating *Aceptable* from *Buena*) is unreliable, and convergence failures or degenerate probability distributions result. This is the primary driver of convergence failures in the 5-class runs and explains why the model assigns zero probability to *Buena* in nearly all folds.